# mamba_va (CompSSM) — Direct dry→wet Compressor Modelling

Trains the **streaming selective state-space model** from `04_s6_ssm/mamba_va`
(`CompSSM`) to reproduce the SSL G-Bus compressor *directly in the waveform
domain*.  This is the analogue of `03_initial_GR_pred/train_lstm_nocond.ipynb`,
but the modelling target is different.

## What changed vs. the GR-prediction LSTM

| | Frame-rate LSTM | mamba_va `CompSSM` |
|---|---|---|
| **Target** | gain-reduction envelope (dB) | the **wet audio** itself |
| **Tokenization** | strided-conv frames (~172 Hz) | none — scalar sample stream |
| **Output** | GR curve, upsampled | `y = u · 10^(g/20)` (multiplicative gain) |
| **Memory** | LSTM hidden state | linear SSM **+** nonlinear attack/release detector |
| **Loss** | L1 on GR (dB) + diff | ESR + pre-emph ESR + multi-res STFT + DC |
| **Eval** | predicted vs target GR | **input vs output audio** (waveform/scatter/listen) |

## Constraints kept from the LSTM notebook

- **No conditioning** (`n_params=0`) — one fixed parameter setting.
- **Same dataset / setting** — Diff-SSL-G-Comp,
  `threshold_-4_attack_1_release_0.4_ratio_10` (the wet renderings the model is
  meant to reproduce).
- Logging via TensorBoard + CSV, checkpoints to Drive.

## Training regime — stateful TBPTT

`CompSSM` is a *streaming, stateful* model: its nonlinear level detector and the
SSM carry the program-dependent, multi-second release memory this setting needs.
So instead of independent crops (which cold-start that memory every chunk), we
walk **contiguous segments** and **carry state across consecutive chunks**
(truncated back-prop-through-time), exactly the regime `mamba_va/train.py` uses.
The streaming, state-exact `render()` path is exercised at evaluation time.

**Loss** is `mamba_va.losses.CombinedLoss` — the model author's recommended
objective (see `mamba_va/DESIGN.md` §2): ESR + **pre-emphasis ESR** (transients)
+ **multi-resolution STFT** (spectral match) + DC.

**Runtime**: select **GPU** via *Runtime → Change runtime type*.

In [ ]:
# ── 0. Install dependencies ──────────────────────────────────────────
# mamba_va itself needs only torch / numpy / soundfile / pyyaml.  We add
# Lightning + TensorBoard for the training loop and logging (as in the LSTM nb).
!pip install -q lightning torchmetrics soundfile pyyaml

In [ ]:
# ── 1. Mount Google Drive & locate dataset ───────────────────────────
import os

try:
    from google.colab import drive
    IN_COLAB = True
except Exception:
    IN_COLAB = False

SETTING = "threshold_-4_attack_1_release_0.4_ratio_10"
DRIVE_DATA_ROOT = None

if IN_COLAB:
    drive.mount("/content/drive", force_remount=False)

    import glob as _g
    def _find_dataset_root():
        cands  = _g.glob("/content/drive/*/*/*/Diff-SSL-G-Comp")
        cands += _g.glob("/content/drive/*/*/Diff-SSL-G-Comp")
        cands += _g.glob("/content/drive/*/Diff-SSL-G-Comp")
        for c in cands:
            if (os.path.isdir(os.path.join(c, "processed_normalized"))
                    and os.path.isdir(os.path.join(c, "processed_ground_truth"))):
                return c
        return None
    DRIVE_DATA_ROOT = _find_dataset_root()
else:
    # Local fallbacks (Mac thesis drive, then a generic ~/data location).
    for c in ["/Volumes/Saola's Drive/AllCode/thesis/data/Diff-SSL-G-Comp",
              os.path.expanduser("~/data/Diff-SSL-G-Comp")]:
        if os.path.isdir(c):
            DRIVE_DATA_ROOT = c
            break

DATA_ROOT = DRIVE_DATA_ROOT
assert DATA_ROOT and os.path.isdir(DATA_ROOT), (
    "Diff-SSL-G-Comp not found. Set DRIVE_DATA_ROOT manually above.")

dry_dir = os.path.join(DATA_ROOT, "processed_normalized")
wet_dir = os.path.join(DATA_ROOT, "processed_ground_truth", SETTING)
assert os.path.isdir(dry_dir), f"Missing dry folder: {dry_dir}"
assert os.path.isdir(wet_dir), f"Missing wet folder: {wet_dir}"

n_dry = len([f for f in os.listdir(dry_dir) if f.endswith(".wav")])
n_wet = len([f for f in os.listdir(wet_dir) if f.endswith("-exported.wav")])
print(f"Dataset root : {DATA_ROOT}")
print(f"Setting      : {SETTING}")
print(f"Dry files    : {n_dry}")
print(f"Wet files    : {n_wet}   (*-exported.wav)")

OUTPUT_DIR = os.path.join(os.path.dirname(DATA_ROOT), "mamba_va_runs")
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Outputs will be saved to: {OUTPUT_DIR}")

In [ ]:
# ── 1b. Cache dry + wet WAVs to local SSD (Colab) ────────────────────
# Unlike the GR notebook (which cached .pt envelopes), CompSSM trains on the
# raw wet audio, so we mirror both the dry inputs and the wet renderings for
# every song that has a rendering under SETTING.
import shutil, time
from pathlib import Path

LOCAL_DATA_ROOT = "/content/Diff-SSL-G-Comp"
USE_LOCAL_CACHE = IN_COLAB

def _robust_copy(src: Path, dst: Path, max_retries: int = 5):
    for attempt in range(1, max_retries + 1):
        try:
            with open(src, "rb") as fsrc, open(dst, "wb") as fdst:
                shutil.copyfileobj(fsrc, fdst, length=1024 * 1024)
            return
        except OSError as e:
            print(f"  [retry {attempt}/{max_retries}] {src.name}: {e}")
            try: dst.unlink(missing_ok=True)
            except Exception: pass
            time.sleep(2 * attempt)
    raise RuntimeError(f"Failed to copy {src} after {max_retries} retries")

def _mirror(src: Path, dst: Path):
    if not dst.exists() or dst.stat().st_size != src.stat().st_size:
        _robust_copy(src, dst)
        return True
    return False

if USE_LOCAL_CACHE:
    wet_src_dir = Path(wet_dir)
    wet_files = sorted(wet_src_dir.glob("*-exported.wav"))
    songs = [p.name.replace("-exported.wav", "") for p in wet_files]
    print(f"Found {len(songs)} songs with wet renderings for '{SETTING}'.")

    local_dry = Path(LOCAL_DATA_ROOT) / "processed_normalized"
    local_wet = Path(LOCAL_DATA_ROOT) / "processed_ground_truth" / SETTING
    local_dry.mkdir(parents=True, exist_ok=True)
    local_wet.mkdir(parents=True, exist_ok=True)

    for i, song in enumerate(songs, 1):
        dry_src = Path(dry_dir) / f"{song}_UnmasteredWAV.wav"
        wet_src = wet_src_dir   / f"{song}-exported.wav"
        if not dry_src.exists():
            print(f"  WARNING: no dry file for '{song}' — skipping")
            continue
        c1 = _mirror(dry_src, local_dry / dry_src.name)
        c2 = _mirror(wet_src, local_wet / wet_src.name)
        flag = "" if (c1 or c2) else "  [skip]"
        print(f"  {i}/{len(songs)}  {song}{flag}")

    DATA_ROOT = LOCAL_DATA_ROOT
    dry_dir = str(local_dry)
    wet_dir = str(local_wet)
    print(f"\nUsing local cache: {DATA_ROOT}")
else:
    print(f"Reading directly from: {DATA_ROOT}")

In [ ]:
# ── 1c. Make the mamba_va package importable ─────────────────────────
# Resolution order:
#   1. already importable (installed / on PYTHONPATH)
#   2. sibling source tree  04_s6_ssm/mamba_va/  (local runs)
#   3. on Drive             .../04_s6_ssm/mamba_va/  (Colab + synced repo)
#   4. git clone the repo   (Colab, requires 04_s6_ssm pushed)
import sys, subprocess
from pathlib import Path

def _pkg_root_under(base: Path):
    for c in (base / "mamba_va", base / "04_s6_ssm" / "mamba_va"):
        if (c / "mamba_va" / "__init__.py").is_file():
            return c
    return None

def _ensure_mamba_va():
    try:
        import mamba_va  # noqa: F401
        return Path(mamba_va.__file__).resolve().parents[1]
    except Exception:
        pass

    # (2) search the local tree upwards from CWD
    cwd = Path.cwd()
    for base in [cwd, *cwd.parents]:
        root = _pkg_root_under(base)
        if root is not None:
            sys.path.insert(0, str(root))
            return root

    # (3) search Google Drive (Colab)
    if IN_COLAB:
        import glob as _g
        hits = _g.glob("/content/drive/**/04_s6_ssm/mamba_va/mamba_va/__init__.py",
                       recursive=True)
        if hits:
            root = Path(hits[0]).resolve().parents[1]
            sys.path.insert(0, str(root))
            return root

        # (4) clone the repo
        repo = "/content/var-repo"
        if not os.path.isdir(repo):
            subprocess.run(
                ["git", "clone", "--depth", "1",
                 "https://github.com/5aola/Virtual-Analogue-Compressor-Modelling.git",
                 repo], check=True)
        root = _pkg_root_under(Path(repo))
        if root is not None:
            sys.path.insert(0, str(root))
            return root
    return None

_root = _ensure_mamba_va()
import mamba_va
from mamba_va import CompSSM
from mamba_va.losses import CombinedLoss, esr
from mamba_va.utils import detach_state
print(f"mamba_va {mamba_va.__version__} from {Path(mamba_va.__file__).parent}")

In [ ]:
# ── 2. Dataset & DataModule — stateful TBPTT over (dry, wet) audio ────
#
# CompSSM is a streaming, stateful model: the nonlinear level detector and the
# SSM carry the program-dependent, multi-second release memory this setting
# needs.  Independent crops cold-start that memory every chunk, so we instead
# walk *contiguous* segments and carry state across consecutive chunks
# (truncated BPTT) — the regime mamba_va/train.py uses.
#
# A batch is a group of recordings advanced through time together.  Each item is
# a dict {x:(B,seq), y:(B,seq), reset:bool}; `reset` marks the first chunk of a
# segment, telling the training loop to reinitialise the model state.
import os, glob, math
import numpy as np
import soundfile as sf
import torch
import lightning as pl
from torch.utils.data import IterableDataset, DataLoader
from typing import Optional

SAMPLE_RATE        = 44100
TBPTT_SEQ_LEN      = 4096   # truncation / gradient window (>=2048 keeps MR-STFT scales)
SEGMENT_CHUNKS     = 64     # chunks per contiguous walk -> ~5.9 s of carried state
PASSES_PER_EPOCH   = 6      # random-segment re-draws per epoch (raises epoch size)
VAL_SEGMENT_CHUNKS = 96     # deterministic val walk length per song

DEVICE = ("cuda" if torch.cuda.is_available()
          else "mps" if getattr(torch.backends, "mps", None)
          and torch.backends.mps.is_available() else "cpu")
print("device:", DEVICE)


def _load_mono(path: str, sr_expected: int = SAMPLE_RATE) -> np.ndarray:
    x, sr = sf.read(path, dtype="float32", always_2d=True)
    x = x.mean(axis=1)  # downmix to mono
    if sr != sr_expected:
        import torchaudio
        x = torchaudio.functional.resample(torch.from_numpy(x), sr, sr_expected).numpy()
    return np.ascontiguousarray(x)


def discover_pairs(dry_dir: str, wet_dir: str):
    dry_lookup = {}
    for p in sorted(glob.glob(os.path.join(dry_dir, "*_UnmasteredWAV.wav"))):
        song = os.path.basename(p).replace("_UnmasteredWAV.wav", "")
        dry_lookup[song] = p
    pairs = []
    for wp in sorted(glob.glob(os.path.join(wet_dir, "*-exported.wav"))):
        song = os.path.basename(wp).replace("-exported.wav", "")
        if song in dry_lookup:
            pairs.append((song, dry_lookup[song], wp))
    return pairs


class TBPTTAudioDataset(IterableDataset):
    # Yields contiguous chunks with carried state.  Each epoch makes `passes`
    # sweeps; per sweep, recordings are shuffled into groups of `batch_size`, a
    # random contiguous segment of `segment_chunks * seq_len` samples is drawn
    # per recording, and the group is walked in `seq_len` steps (state carried,
    # `reset=True` on the first step).

    def __init__(self, songs, cache, seq_len, batch_size, segment_chunks,
                 passes=1, shuffle=True, drop_last=True, seed=None):
        self.songs = list(songs)
        self.cache = cache
        self.seq_len = seq_len
        self.batch_size = min(batch_size, len(self.songs))
        self.segment_chunks = segment_chunks
        self.passes = passes
        self.shuffle = shuffle
        self.drop_last = drop_last
        self.seed = seed

    def _n_groups(self):
        if self.drop_last:
            return len(self.songs) // self.batch_size
        return math.ceil(len(self.songs) / self.batch_size)

    def __len__(self):
        return self.passes * self._n_groups() * self.segment_chunks

    def __iter__(self):
        rng = np.random.default_rng(self.seed)   # seed=None -> fresh each epoch
        seg = self.segment_chunks * self.seq_len
        for _ in range(self.passes):
            order = list(self.songs)
            if self.shuffle:
                rng.shuffle(order)
            for gi in range(0, len(order), self.batch_size):
                grp = order[gi:gi + self.batch_size]
                if self.drop_last and len(grp) < self.batch_size:
                    continue
                Xs, Ys = [], []
                for s in grp:
                    d, w = self.cache[s]
                    T = len(d)
                    if T > seg:
                        start = int(rng.integers(0, T - seg + 1)) if self.shuffle else 0
                        dd, ww = d[start:start + seg], w[start:start + seg]
                    else:
                        pad = seg - T
                        dd, ww = np.pad(d, (0, pad)), np.pad(w, (0, pad))
                    Xs.append(dd); Ys.append(ww)
                X = np.stack(Xs); Y = np.stack(Ys)       # (B, seg)
                for c in range(self.segment_chunks):
                    t = c * self.seq_len
                    yield {
                        "x": torch.from_numpy(np.ascontiguousarray(X[:, t:t + self.seq_len])),
                        "y": torch.from_numpy(np.ascontiguousarray(Y[:, t:t + self.seq_len])),
                        "reset": c == 0,
                    }


class AudioPairDataModule(pl.LightningDataModule):
    def __init__(self, dry_dir, wet_dir, seq_len=TBPTT_SEQ_LEN, sample_rate=SAMPLE_RATE,
                 train_split=0.8, batch_size=8, segment_chunks=SEGMENT_CHUNKS,
                 passes_per_epoch=PASSES_PER_EPOCH, val_segment_chunks=VAL_SEGMENT_CHUNKS):
        super().__init__()
        self.dry_dir = dry_dir
        self.wet_dir = wet_dir
        self.seq_len = seq_len
        self.sample_rate = sample_rate
        self.train_split = train_split
        self.batch_size = batch_size
        self.segment_chunks = segment_chunks
        self.passes_per_epoch = passes_per_epoch
        self.val_segment_chunks = val_segment_chunks
        self.cache = None

    def setup(self, stage: Optional[str] = None) -> None:
        if self.cache is not None:
            return
        pairs = discover_pairs(self.dry_dir, self.wet_dir)
        if len(pairs) < 2:
            raise ValueError("Need >= 2 songs for a train/val split.")

        self.cache = {}
        for song, dp, wp in pairs:
            d = _load_mono(dp, self.sample_rate)
            w = _load_mono(wp, self.sample_rate)
            L = min(len(d), len(w))
            self.cache[song] = (d[:L], w[:L])

        songs = sorted(self.cache.keys())
        g = torch.Generator().manual_seed(42)
        perm = torch.randperm(len(songs), generator=g).tolist()
        n_train = min(max(1, int(len(songs) * self.train_split)), len(songs) - 1)
        self.train_songs = [songs[i] for i in perm[:n_train]]
        self.val_songs = [s for s in songs if s not in self.train_songs]

        self.train_ds = TBPTTAudioDataset(
            self.train_songs, self.cache, self.seq_len, self.batch_size,
            self.segment_chunks, passes=self.passes_per_epoch,
            shuffle=True, drop_last=True, seed=None)
        self.val_ds = TBPTTAudioDataset(
            self.val_songs, self.cache, self.seq_len, len(self.val_songs),
            self.val_segment_chunks, passes=1,
            shuffle=False, drop_last=False, seed=0)

        ctx_s = self.segment_chunks * self.seq_len / self.sample_rate
        mb = sum((d.nbytes + w.nbytes) for d, w in self.cache.values()) / 1024 / 1024
        print(f"Cached {len(self.cache)} songs ({mb:.0f} MB).")
        print(f"Train: {len(self.train_ds)} chunks/epoch "
              f"({self.passes_per_epoch} passes x {len(self.train_ds) // max(1, self.passes_per_epoch)} chunks) "
              f"from {len(self.train_songs)} songs {self.train_songs}")
        print(f"Val:   {len(self.val_ds)} chunks "
              f"from {len(self.val_songs)} songs {self.val_songs}")
        print(f"Carried context per walk: {self.segment_chunks} x {self.seq_len} "
              f"= {ctx_s:.1f} s")

    def train_dataloader(self):
        return DataLoader(self.train_ds, batch_size=None, num_workers=0, pin_memory=True)

    def val_dataloader(self):
        return DataLoader(self.val_ds, batch_size=None, num_workers=0, pin_memory=True)

In [ ]:
# ── 3. Lightning system wrapping CompSSM (stateful TBPTT) ────────────
#
# State is carried across training_steps and reset on `reset` (the first chunk
# of each segment).  Gradients are truncated at the chunk boundary by detaching
# the carried state, so back-prop stays O(seq_len) while the *forward* memory
# spans the whole segment.
#
# Loss = ESR + pre-emph ESR + multi-res STFT + DC  (mamba_va.losses.CombinedLoss,
# the objective recommended in mamba_va/DESIGN.md §2).  Raw ESR is also logged
# as an interpretable metric (0 = perfect, 1 = predicting silence).
from lightning.pytorch.callbacks import ModelCheckpoint, LearningRateMonitor


class CompSSMSystem(pl.LightningModule):
    def __init__(self, model, lr=3e-4, w_esr=1.0, w_preemph=0.5, w_stft=0.5,
                 w_dc=0.1, lr_patience=15, min_lr=1e-6):
        super().__init__()
        self.model = model
        self.crit = CombinedLoss(w_esr=w_esr, w_preemph=w_preemph,
                                 w_stft=w_stft, w_dc=w_dc)
        self.lr = lr
        self.lr_patience = lr_patience
        self.min_lr = min_lr
        self._train_state = None
        self._val_state = None

    def on_train_epoch_start(self):
        self._train_state = None

    def on_validation_epoch_start(self):
        self._val_state = None

    def _run(self, batch, state):
        if batch["reset"]:
            state = None
        y, state = self.model(batch["x"], state=state, parallel=True)
        state = detach_state(state)               # truncate BPTT at chunk edge
        loss = self.crit(y, batch["y"])
        with torch.no_grad():
            e = esr(y, batch["y"])
        return loss, e, state

    def training_step(self, batch, batch_idx):
        loss, e, self._train_state = self._run(batch, self._train_state)
        self.log("loss/train", loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log("esr/train", e, on_step=False, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        loss, e, self._val_state = self._run(batch, self._val_state)
        self.log("loss/val", loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log("esr/val", e, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def configure_optimizers(self):
        opt = torch.optim.AdamW(self.parameters(), lr=self.lr)
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
            opt, mode="min", factor=0.5, patience=self.lr_patience, min_lr=self.min_lr)
        return {"optimizer": opt,
                "lr_scheduler": {"scheduler": sched, "monitor": "loss/val"}}

In [ ]:
# ── 4. Build model + run config ──────────────────────────────────────
import time, json, hashlib
from datetime import datetime
from lightning.pytorch.callbacks import EarlyStopping, TQDMProgressBar
from lightning.pytorch.loggers import TensorBoardLogger, CSVLogger

torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")

# ╔══════════════════════════════════════════════════════════════════╗
# ║ RESUME                                                            ║
# ╚══════════════════════════════════════════════════════════════════╝
RESUME_RUN: str | None = None
RUN_TAG: str = "mamba_va_nocond_tbptt"

# ── hparams ──────────────────────────────────────────────────────────
BATCH_SIZE          = 8       # recordings advanced together (>= n train songs -> 1 group)
LR                  = 3e-4
MAX_EPOCHS          = 300
EARLY_STOP_PATIENCE = 40
LR_PLATEAU_PATIENCE = 15

# TBPTT walk (defined in cell 2): TBPTT_SEQ_LEN, SEGMENT_CHUNKS,
# PASSES_PER_EPOCH, VAL_SEGMENT_CHUNKS.

# CompSSM (no conditioning -> N_PARAMS = 0)
N_PARAMS     = 0
D_MODEL      = 24
D_STATE      = 16
N_LAYERS     = 3
EXPAND       = 2
CONV_KERNEL  = 4
N_BANDS      = 4
MAX_DB       = 24.0     # gain bound; compression only needs a few dB

# loss weights (mamba_va.losses.CombinedLoss; DESIGN.md §2 defaults)
W_ESR, W_PREEMPH, W_STFT, W_DC = 1.0, 0.5, 0.5, 0.1
# ─────────────────────────────────────────────────────────────────────

if RESUME_RUN:
    RUN_NAME = RESUME_RUN
    RUN_DIR = os.path.join(OUTPUT_DIR, RUN_NAME)
    _resume_ckpt = os.path.join(RUN_DIR, "checkpoints", "last.ckpt")
    assert os.path.isfile(_resume_ckpt), f"No last.ckpt in {RUN_DIR}"
    print(f"RESUMING run: {RUN_NAME}")
else:
    _ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    RUN_NAME = f"mamba_va_{_ts}_{RUN_TAG}" if RUN_TAG else f"mamba_va_{_ts}"
    RUN_DIR = os.path.join(OUTPUT_DIR, RUN_NAME)
    _resume_ckpt = None
    print(f"NEW run: {RUN_NAME}")

os.makedirs(RUN_DIR, exist_ok=True)
print(f"Run dir: {RUN_DIR}")
if IN_COLAB:
    assert "/drive/" not in DATA_ROOT, "DATA_ROOT still on Drive — run cell 1b first."

dm = AudioPairDataModule(
    dry_dir=dry_dir, wet_dir=wet_dir,
    seq_len=TBPTT_SEQ_LEN, sample_rate=SAMPLE_RATE,
    train_split=0.8, batch_size=BATCH_SIZE,
    segment_chunks=SEGMENT_CHUNKS, passes_per_epoch=PASSES_PER_EPOCH,
    val_segment_chunks=VAL_SEGMENT_CHUNKS,
)

model = CompSSM(
    n_params=N_PARAMS, d_model=D_MODEL, d_state=D_STATE, n_layers=N_LAYERS,
    expand=EXPAND, conv_kernel=CONV_KERNEL, n_bands=N_BANDS, max_db=MAX_DB,
)
n_params = model.num_params()
print(f"CompSSM parameters : {n_params:,}  (conditioning n_params = {N_PARAMS})")
print(f"TBPTT seq_len      : {TBPTT_SEQ_LEN} samples "
      f"({TBPTT_SEQ_LEN / SAMPLE_RATE * 1000:.0f} ms gradient window)")

# ── save hparams ─────────────────────────────────────────────────────
_hparams = {
    "sample_rate": SAMPLE_RATE,
    "regime": "stateful_tbptt",
    "tbptt": {"seq_len": TBPTT_SEQ_LEN, "segment_chunks": SEGMENT_CHUNKS,
              "passes_per_epoch": PASSES_PER_EPOCH,
              "val_segment_chunks": VAL_SEGMENT_CHUNKS},
    "setting": SETTING,
    "batch_size": BATCH_SIZE,
    "split_unit": "song", "split_seed": 42, "train_split": 0.8,
    "lr": LR, "max_epochs": MAX_EPOCHS,
    "early_stop_patience": EARLY_STOP_PATIENCE,
    "lr_plateau_patience": LR_PLATEAU_PATIENCE,
    "model_type": "mamba_va.CompSSM",
    "loss": {"kind": "esr+preemph_esr+mrstft+dc",
             "w_esr": W_ESR, "w_preemph": W_PREEMPH,
             "w_stft": W_STFT, "w_dc": W_DC},
    "compssm": {"n_params": N_PARAMS, "d_model": D_MODEL, "d_state": D_STATE,
                "n_layers": N_LAYERS, "expand": EXPAND,
                "conv_kernel": CONV_KERNEL, "n_bands": N_BANDS,
                "max_db": MAX_DB, "num_params": n_params},
}
with open(os.path.join(RUN_DIR, "hparams.json"), "w") as _f:
    json.dump(_hparams, _f, indent=2)
print("Saved hparams.json")

system = CompSSMSystem(
    model=model, lr=LR, w_esr=W_ESR, w_preemph=W_PREEMPH,
    w_stft=W_STFT, w_dc=W_DC, lr_patience=LR_PLATEAU_PATIENCE,
)

# ── callbacks & loggers ──────────────────────────────────────────────
ckpt_dir = os.path.join(RUN_DIR, "checkpoints")
best_cb = ModelCheckpoint(
    dirpath=ckpt_dir, monitor="loss/val", mode="min",
    save_top_k=3, save_last=True, filename="best-{epoch:03d}-{step}",
    auto_insert_metric_name=False)
periodic_cb = ModelCheckpoint(
    dirpath=ckpt_dir, every_n_epochs=5, save_top_k=-1,
    filename="epoch-{epoch:03d}", auto_insert_metric_name=False)
lr_cb = LearningRateMonitor(logging_interval="epoch")
early_stop_cb = EarlyStopping(
    monitor="loss/val", mode="min", patience=EARLY_STOP_PATIENCE,
    min_delta=0.0, verbose=True)

tb_logger = TensorBoardLogger(save_dir=RUN_DIR, name="tb", version="")
csv_logger = CSVLogger(save_dir=RUN_DIR, name="csv", version="")

In [ ]:
# ── 5. Launch TensorBoard & fit ──────────────────────────────────────
if IN_COLAB:
    _tb_link = "/content/tb_current"
    if os.path.islink(_tb_link) or os.path.exists(_tb_link):
        os.remove(_tb_link)
    os.symlink(os.path.join(RUN_DIR, "tb"), _tb_link)
    %load_ext tensorboard
    %tensorboard --logdir /content/tb_current

trainer = pl.Trainer(
    max_epochs=MAX_EPOCHS,
    accelerator="auto",
    devices="auto",
    precision="32-true",          # custom scan + detector → keep full precision
    callbacks=[best_cb, periodic_cb, lr_cb, early_stop_cb,
               TQDMProgressBar(refresh_rate=5)],
    logger=[tb_logger, csv_logger],
    log_every_n_steps=10,
    default_root_dir=RUN_DIR,
    gradient_clip_val=1.0,
)

t0 = time.time()
trainer.fit(system, dm, ckpt_path=_resume_ckpt)
print(f"\nTotal training time: {(time.time() - t0) / 60:.1f} min")
print(f"Best val loss: {best_cb.best_model_score:.4f}")
print(f"Best ckpt    : {best_cb.best_model_path}")

In [ ]:
# ── 6. Evaluate: generic input → output comparison (no GR curves) ────
#
# Streams a loud excerpt of each validation song through the trained model and
# compares the *audio* it produces with the target wet audio:
#   • waveform overlay (dry / target / predicted) on a short zoom
#   • prediction error over the excerpt
#   • input-vs-output sample scatter — the compression characteristic
#   • ESR per excerpt, and listenable audio for dry / target / predicted
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Audio, display

# best checkpoint → system
ckpt = torch.load(best_cb.best_model_path, map_location=DEVICE, weights_only=False)
system.load_state_dict(ckpt["state_dict"])
system.eval().to(DEVICE)
print(f"Loaded best checkpoint: {best_cb.best_model_path}")

dm.setup()
EVAL_SECONDS = 6.0
n = int(EVAL_SECONDS * SAMPLE_RATE)


def _loud_start(x, n):
    if len(x) <= n:
        return 0
    step = max(1, n // 2)
    best, bestv = 0, -1.0
    for s in range(0, len(x) - n, step):
        v = float(np.sqrt(np.mean(x[s:s + n] ** 2)))
        if v > bestv:
            bestv, best = v, s
    return best


def _esr_np(pred, target, eps=1e-8):
    return float(np.sum((target - pred) ** 2) / (np.sum(target ** 2) + eps))


results = []
for song in dm.val_songs:
    dry, wet = dm.cache[song]
    start = _loud_start(dry, n)
    u = torch.from_numpy(dry[start:start + n]).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        y, _ = system.model(u, parallel=True)       # streaming-equivalent pass
    pred = y.squeeze(0).float().cpu().numpy()
    results.append({
        "song": song, "start": start,
        "dry": dry[start:start + n], "target": wet[start:start + n], "pred": pred,
        "esr": _esr_np(pred, wet[start:start + n]),
    })

# ── plots ────────────────────────────────────────────────────────────
nrows = len(results)
fig, axes = plt.subplots(nrows, 3, figsize=(18, 3.2 * nrows), squeeze=False)
zoom = int(0.08 * SAMPLE_RATE)          # 80 ms waveform zoom
zoff = n // 2
t_zoom = (np.arange(zoom)) / SAMPLE_RATE * 1000  # ms

for r, res in enumerate(results):
    d, tg, pr = res["dry"], res["target"], res["pred"]

    ax = axes[r][0]
    ax.plot(t_zoom, d[zoff:zoff + zoom], color="gray", alpha=0.5, lw=0.8, label="dry in")
    ax.plot(t_zoom, tg[zoff:zoff + zoom], color="tab:green", lw=0.9, label="target out")
    ax.plot(t_zoom, pr[zoff:zoff + zoom], color="tab:orange", lw=0.9, label="pred out")
    ax.set_title(f"{res['song']} — waveform (80 ms)")
    ax.set_xlabel("ms"); ax.set_ylabel("amp")
    ax.legend(loc="upper right", fontsize=7)

    ax = axes[r][1]
    t_full = np.arange(n) / SAMPLE_RATE
    ax.plot(t_full, pr - tg, color="tab:red", lw=0.4)
    ax.set_title(f"error (pred − target) — ESR={res['esr']:.4f}")
    ax.set_xlabel("s"); ax.set_ylabel("err"); ax.axhline(0, color="gray", lw=0.5, ls="--")

    ax = axes[r][2]
    k = slice(None, None, max(1, n // 4000))     # subsample for the scatter
    ax.scatter(d[k], pr[k], s=2, alpha=0.25, color="tab:blue", label="pred")
    ax.scatter(d[k], tg[k], s=2, alpha=0.15, color="tab:green", label="target")
    lim = float(max(np.abs(d).max(), 1e-3))
    ax.plot([-lim, lim], [-lim, lim], color="gray", lw=0.6, ls="--", label="unity")
    ax.set_title("input vs output (compression curve)")
    ax.set_xlabel("dry in"); ax.set_ylabel("out"); ax.legend(loc="upper left", fontsize=7)

fig.suptitle(f"mamba_va CompSSM — best val loss {best_cb.best_model_score:.4f}", y=1.005)
fig.tight_layout()
plot_path = os.path.join(RUN_DIR, "eval_io_comparison.png")
fig.savefig(plot_path, dpi=150, bbox_inches="tight")
print(f"Saved plot → {plot_path}")
plt.show()

print("\nPer-song ESR (lower = better):")
for res in results:
    print(f"  {res['song']:<20s}  ESR = {res['esr']:.4f}")

# ── save + listen ────────────────────────────────────────────────────
for res in results:
    base = os.path.join(RUN_DIR, f"eval_{res['song']}")
    sf.write(base + "_dry.wav",    res["dry"].astype(np.float32),    SAMPLE_RATE)
    sf.write(base + "_target.wav", res["target"].astype(np.float32), SAMPLE_RATE)
    sf.write(base + "_pred.wav",   res["pred"].astype(np.float32),   SAMPLE_RATE)

res0 = results[0]
print(f"\nListen — {res0['song']}:  dry / target / predicted")
display(Audio(res0["dry"],    rate=SAMPLE_RATE))
display(Audio(res0["target"], rate=SAMPLE_RATE))
display(Audio(res0["pred"],   rate=SAMPLE_RATE))